In [ ]:
# import packages
import pandas as pd
import numpy as np
import os
import requests
import plotly.express as px

os.environ["PUDL_DATA_STORE"] = "s3"
os.environ["PUDL_BUILD"] = "stable"

%reload_ext autoreload
%autoreload 2

year = 2025

## Scope 2 Analysis
How important is consumption-based: role of imports across regions. Look at gross imports
How important is spatial granularity: look at BA, region, national

### Question 1: role of imports
How much of regional load is generally served by imports
Data/methods:
1. download INTERCHANGE data
2. Calculate gross imports and gross exports for each region
3. Group by BA-year
4. Merge with balance data
5. % imports = imports_mwh / (demand_mwh + exports_mwh)

This could be evaluated from an emissions perspective as well:
CO2 imported / (CO2 consumed + CO2 exported)

### Question 2: Hierarchy ordering
How should spatial granularity / temporal granularity / type be prioritized. 
We want to calculate emission rates based on each method, then calculate total annual emissions by multiplying the EF by the demand in each hour and summing
We can then calculate an annual (and hourly) error rate, treating the hourly/BA/consumed factors as "truth"

Start with a single BA (MISO), then expand analysis to all

Data methods:
1. Download the BA file, filtering to a specific year of data
2. Add month/year and region columns
3. Calculate rollups: sum emissions and MWh and use to recalculate and transform to hourly
4. Calculate interconnection factors - need to sum across all BAs/regions in an interconnection, and roll these up
5. Get national factors for reference (US48 file)
6. Calculate hourly emissions by multiplying demand by EF
7. Calculate error rates
8. Get annual average error metrics, separated by category
9. Sort by different ranking and see how error rates decline 


How to compare: 
- For each BA-hour, calculate the following factors
    - BA_hour_consumed
    - ba_hour_produced
    - ba_month_consumed
    - ba_month_produced
    - ba_year_consumed
    - ba_year_produced
    - grid_hour_consumed
    - grid_hour_produced
    - grid_month_consumed
    - grid_month_produced
    - grid_year_consumed
    - grid_year_produced
    - national_hour_consumed
    - national_hour_produced
    - national_month_consumed
    - national_month_produced
    - national_year_consumed
    - national_year_produced

Spatial aggregations:
- BA: operational grid boundary
- Region: could be example of large vs small BAs
- Interconnection: grid-wide
- National: example of today


### Question 3: importance of local regions
Do "local" subdivisions provide meaningful gain in accuracy. 

Test 1: MISO
- Use data from MISO dashboard for LRZs, and LRZ demand data from EIA-930
- Use that as the source of truth, and compare to MISO-wide numbers

Test 2: EIA "regions" as proxy for large regions
- Examples: Northwest, Florida, Southwest, Carolinas



In [ ]:
# region and columns
ba_codes = [
    "AECI",
    "AVA",
    "AVRN",
    "AZPS",
    "BANC",
    "BHBA",
    "BPAT",
    "CHPD",
    "CISO",
    "CPLE",
    "CPLW",
    "DEAA",
    "DOPD",
    "DUK",
    "EEI",
    "EPE",
    "ERCO",
    "FMPP",
    "FPC",
    "FPL",
    "GCPD",
    "GLHB",
    "GRID",
    "GRIF",
    "GRMA",
    "GVL",
    "GWA",
    "HGMA",
    "HST",
    "IID",
    "IPCO",
    "ISNE",
    "JEA",
    "LDWP",
    "LGEE",
    "MISO",
    "NEVP",
    "NSB",
    "NWMT",
    "NYIS",
    "OVEC",
    "PACE",
    "PACW",
    "PGE",
    "PJM",
    "PNM",
    "PSCO",
    "PSEI",
    "SC",
    "SCEG",
    "SCL",
    "SEC",
    "SEPA",
    "SIKE",
    "SOCO",
    "SPA",
    "SRP",
    "SWPP",
    "SWPW",
    "TAL",
    "TEC",
    "TEPC",
    "TIDC",
    "TPWR",
    "TVA",
    "WACM",
    "WALC",
    "WAUE",
    "WAUW",
    "WWA",
    "YAD",
    "AESO",
    "BCHA",
    "HQT",
    "IESO",
    "MHEB",
    "NBSO",
    "SPC",
    "CEN",
    "CFE",
]

region_codes = [
    "CAL",
    "CAR",
    "CENT",
    "FLA",
    "MIDA",
    "MIDW",
    "NE",
    "NW",
    "NY",
    "SE",
    "SW",
    "TEN",
    "TEX",
    "US48",
    "CAN",
    "MEX",
]

ba_to_region = {
    "AEC": "SE",
    "AECI": "MIDW",
    "AVA": "NW",
    "AVRN": "NW",
    "AZPS": "SW",
    "BANC": "CAL",
    "BHBA": "NW",
    "BPAT": "NW",
    "CHPD": "NW",
    "CISO": "CAL",
    "CPLE": "CAR",
    "CPLW": "CAR",
    "DEAA": "SW",
    "DOPD": "NW",
    "DUK": "CAR",
    "EEI": "MIDW",
    "EPE": "SW",
    "ERCO": "TEX",
    "FMPP": "FLA",
    "FPC": "FLA",
    "FPL": "FLA",
    "GCPD": "NW",
    "GLHB": "MIDW",
    "GRID": "NW",
    "GRIF": "SW",
    "GRMA": "SW",
    "GVL": "FLA",
    "GWA": "NW",
    "HGMA": "SW",
    "HST": "FLA",
    "IID": "CAL",
    "IPCO": "NW",
    "ISNE": "NE",
    "JEA": "FLA",
    "LDWP": "CAL",
    "LGEE": "MIDW",
    "MISO": "MIDW",
    "NEVP": "NW",
    "NSB": "FLA",
    "NWMT": "NW",
    "NYIS": "NY",
    "OVEC": "MIDA",
    "PACE": "NW",
    "PACW": "NW",
    "PGE": "NW",
    "PJM": "MIDA",
    "PNM": "SW",
    "PSCO": "NW",
    "PSEI": "NW",
    "SC": "CAR",
    "SCEG": "CAR",
    "SCL": "NW",
    "SEC": "FLA",
    "SEPA": "SE",
    "SIKE": "MIDW",
    "SOCO": "SE",
    "SPA": "CENT",
    "SRP": "SW",
    "SWPP": "CENT",
    "SWPW": "NW",
    "TAL": "FLA",
    "TEC": "FLA",
    "TEPC": "SW",
    "TIDC": "CAL",
    "TPWR": "NW",
    "TVA": "TEN",
    "WACM": "NW",
    "WALC": "SW",
    "WAUE": "NW",
    "WAUW": "NW",
    "WWA": "NW",
    "YAD": "CAR",
    "AESO": "CAN",
    "BCHA": "CAN",
    "HQT": "CAN",
    "IESO": "CAN",
    "MHEB": "CAN",
    "NBSO": "CAN",
    "SPC": "CAN",
    "CEN": "MEX",
    "CFE": "MEX",
}

region_to_grid = {
    "CAL": "Western",
    "CAR": "Eastern",
    "CENT": "Eastern",
    "FLA": "Eastern",
    "MIDA": "Eastern",
    "MIDW": "Eastern",
    "NE": "Eastern",
    "NW": "Western",
    "NY": "Eastern",
    "SE": "Eastern",
    "SW": "Western",
    "TEN": "Eastern",
    "TEX": "Texas",
}

ba_columns_to_use = [
    "BA",
    "UTC time",
    # "Local date",
    # "Hour",
    "Local time",
    # "Time zone",
    # "Generation only?",
    # "Demand forecast",
    # "Demand",
    # "Net generation",
    # "Total interchange",
    # "Imputed demand",
    # "Imputed net generation",
    # "Imputed total interchange",
    "Adjusted demand",
    "Adjusted net generation",
    "Adjusted total interchange",
    # "NG: COL",
    # "NG: NG",
    # "NG: NUC",
    # "NG: OIL",
    # "NG: GEO",
    # "NG: WAT",
    # "NG: PS",
    # "NG: SUN",
    # "NG: SNB",
    # "NG: WND",
    # "NG: WNB",
    # "NG: BAT",
    # "NG: OES",
    # "NG: UES",
    # "NG: OTH",
    # "NG: UNK",
    # "Imputed COL Gen",
    # "Imputed NG Gen",
    # "Imputed NUC Gen",
    # "Imputed OIL Gen",
    # "Imputed GEO Gen",
    # "Imputed WAT Gen",
    # "Imputed PS Gen",
    # "Imputed SUN Gen",
    # "Imputed SNB Gen",
    # "Imputed WND Gen",
    # "Imputed WNB Gen",
    # "Imputed BAT Gen",
    # "Imputed OES Gen",
    # "Imputed UES Gen",
    # "Imputed OTH Gen",
    # "Imputed UNK Gen",
    # "Adjusted COL Gen",
    # "Adjusted NG Gen",
    # "Adjusted NUC Gen",
    # "Adjusted OIL Gen",
    # "Adjusted GEO Gen",
    # "Adjusted WAT Gen",
    # "Adjusted PS Gen",
    # "Adjusted SUN Gen",
    # "Adjusted SNB Gen",
    # "Adjusted WND Gen",
    # "Adjusted WNB Gen",
    # "Adjusted BAT Gen",
    # "Adjusted OES Gen",
    # "Adjusted UES Gen",
    # "Adjusted OTH Gen",
    # "Adjusted UNK Gen",
    # "CO2 Factor: COL",
    # "CO2 Factor: NG",
    # "CO2 Factor: OIL",
    # "CO2 Emissions: COL",
    # "CO2 Emissions: NG",
    # "CO2 Emissions: OIL",
    # "CO2 Emissions: Other",
    "CO2 Emissions Generated",
    "CO2 Emissions Imported",
    "CO2 Emissions Exported",
    "CO2 Emissions Consumed",
    "Positive Generation",
    "Consumed Electricity",
    "CO2 Emissions Intensity for Generated Electricity",
    "CO2 Emissions Intensity for Consumed Electricity",
]

region_columns_to_use = [
    "Region",
    "UTC time",
    # "Local date",
    # "Hour",
    "Local time",
    # "Time zone",
    # "Demand forecast",
    "Demand",
    "Net generation",
    "Total interchange",
    # "Sum (NG)",
    # "NG: COL",
    # "NG: NG",
    # "NG: NUC",
    # "NG: OIL",
    # "NG: GEO",
    # "NG: WAT",
    # "NG: PS",
    # "NG: SUN",
    # "NG: SNB",
    # "NG: WND",
    # "NG: WNB",
    # "NG: BAT",
    # "NG: OES",
    # "NG: UES",
    # "NG: OTH",
    # "NG: UNK",
    # "Sum (Trade)",
    # "Sum (Imports)",
    # "Sum (Exports)",
    # "Balance NG D TI",
    # "Balance TI Trade",
    # "Balance NG",
    # "CO2 Factor: COL",
    # "CO2 Factor: NG",
    # "CO2 Factor: OIL",
    # "Positive Gen from COL",
    # "Positive Gen from NG",
    # "Positive Gen from OIL",
    # "CO2 Emissions: COL",
    # "CO2 Emissions: NG",
    # "CO2 Emissions: OIL",
    # "CO2 Emissions: Other",
    "CO2 Emissions Generated",
    "CO2 Emissions Imported",
    "CO2 Emissions Exported",
    "CO2 Emissions Consumed",
    "Positive Generation",
    "Consumed Electricity",
    "CO2 Emissions Intensity for Generated Electricity",
    "CO2 Emissions Intensity for Consumed Electricity",
]

column_map = {
    "BA": "ba_code",
    "Balancing Authority": "ba_code",
    "UTC time": "datetime_utc",
    "Local time": "datetime_local",
    "Adjusted demand": "demand_mwh",
    "Adjusted net generation": "net_generation_mwh",
    "Adjusted total interchange": "total_interchange_mwh",
    "Positive Generation": "positive_generation_mwh",
    "Consumed Electricity": "consumed_mwh",
    "CO2 Emissions Intensity for Generated Electricity": "co2_rate_generated_lb_per_mwh",
    "CO2 Emissions Intensity for Consumed Electricity": "co2_rate_consumed_lb_per_mwh",
    "CO2 Emissions Generated": "co2_emissions_generated_lb",
    "CO2 Emissions Imported": "co2_emissions_imported_lb",
    "CO2 Emissions Exported": "co2_emissions_exported_lb",
    "CO2 Emissions Consumed": "co2_emissions_consumed_lb",
}

In [ ]:
# load ba data
# download all ba data into single dataframe and standardize
ba_data_cache = f"ba_data_{year}.csv.zip"

if os.path.exists(ba_data_cache):
    print(f"Loading ba_data from {ba_data_cache}")
    ba_data = pd.read_csv(
        ba_data_cache,
        parse_dates=["datetime_utc", "datetime_local"],
    )
else:
    ba_data = []
    for ba_code in ba_codes:
        try:
            b_data = pd.read_excel(
                f"https://www.eia.gov/electricity/gridmonitor/knownissues/xls/{ba_code}.xlsx",
                sheet_name="Published Hourly Data",
                usecols=lambda col: col in (ba_columns_to_use),
            )
        except ValueError:
            print(f"No data found for {ba_code}")
            continue
        # filter data to a specific local year
        b_data = b_data[b_data["Local time"].dt.year == year]
        # rename columns
        b_data.rename(columns=column_map, inplace=True)
        # convert emission rates
        for col in ["co2_rate_generated_lb_per_mwh", "co2_rate_consumed_lb_per_mwh"]:
            if col in b_data.columns:
                b_data[col] = b_data[col] * 1000
        # rename column
        b_data.rename(
            columns={
                "co2_rate_generated_lb_per_mwh": "ba_hour_generated",
                "co2_rate_consumed_lb_per_mwh": "ba_hour_consumed",
            },
            inplace=True,
        )
        # convert emissions totals from metric tons to pounds
        for col in [
            "co2_emissions_generated_lb",
            "co2_emissions_imported_lb",
            "co2_emissions_exported_lb",
            "co2_emissions_consumed_lb",
        ]:
            if col in b_data.columns:
                b_data[col] = b_data[col] * 2204.62
        ba_data.append(b_data)

    ba_data = pd.concat(ba_data)
    ba_data.to_csv(ba_data_cache, index=False)
    print(f"Saved ba_data to {ba_data_cache}")

ba_data

# Imports Analysis

In [ ]:
# download interchange data to evaluate ba imports
interchange = pd.concat(
    [
        pd.read_csv(
            "https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_INTERCHANGE_2025_Jan_Jun.csv",
            usecols=[
                "Balancing Authority",
                "UTC Time at End of Hour",
                "Local Time at End of Hour",
                "Directly Interconnected Balancing Authority",
                "Interchange (MW)",
            ],
        ),
        pd.read_csv(
            "https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_INTERCHANGE_2025_Jul_Dec.csv",
            usecols=[
                "Balancing Authority",
                "UTC Time at End of Hour",
                "Local Time at End of Hour",
                "Directly Interconnected Balancing Authority",
                "Interchange (MW)",
            ],
        ),
    ],
    axis=0,
)
interchange = interchange.rename(columns=column_map)

# calculate imports and exports for each hour
interchange["imports_mw"] = (
    interchange["Interchange (MW)"].abs().where(interchange["Interchange (MW)"] <= 0, 0)
)
interchange["exports_mw"] = interchange["Interchange (MW)"].where(
    interchange["Interchange (MW)"] >= 0, 0
)

# get total annual imports
annual_imports = (
    interchange.groupby("ba_code")[["imports_mw", "exports_mw"]].sum().reset_index()
)
annual_imports

In [ ]:
import_share = (
    ba_data.groupby("ba_code")[["demand_mwh"]]
    .sum()
    .merge(annual_imports, on="ba_code", how="left")
)
import_share["import_share"] = import_share["imports_mw"] / (
    import_share["demand_mwh"] + import_share["exports_mw"]
)
import_share.sort_values("import_share", ascending=False)

In [ ]:
import_e_share = ba_data.groupby("ba_code")[
    [
        "co2_emissions_imported_lb",
        "co2_emissions_exported_lb",
        "co2_emissions_consumed_lb",
    ]
].sum()
import_e_share["import_share"] = import_e_share["co2_emissions_imported_lb"] / (
    import_e_share["co2_emissions_consumed_lb"]
    + import_e_share["co2_emissions_exported_lb"]
)
import_e_share.sort_values("import_share", ascending=False)


# Hierarchy Analysis

In [ ]:
# add category columns
ba_data["year"] = year
ba_data["month"] = ba_data["datetime_local"].dt.month
ba_data["region"] = ba_data["ba_code"].map(ba_to_region)
ba_data["grid"] = ba_data["region"].map(region_to_grid)
ba_data

In [ ]:
# print the number of rows with na values grouped by ba_code
ba_data[
    ba_data[
        [
            "positive_generation_mwh",
            "consumed_mwh",
            "demand_mwh",
            "co2_emissions_consumed_lb",
            "co2_emissions_generated_lb",
        ]
    ]
    .isna()
    .any(axis=1)
].groupby("ba_code").size()

In [ ]:
# calculate various aggregations of data
# drop rows with missing values
ba_data = ba_data[
    ~ba_data[
        [
            "positive_generation_mwh",
            "consumed_mwh",
            "demand_mwh",
            "co2_emissions_consumed_lb",
            "co2_emissions_generated_lb",
        ]
    ]
    .isna()
    .any(axis=1)
].copy()

# ba_month_consumed
ba_data["ba_month_consumed"] = ba_data.groupby(["ba_code", "month"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["ba_code", "month"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# ba_month_generated
ba_data["ba_month_generated"] = ba_data.groupby(["ba_code", "month"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["ba_code", "month"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# ba_year_consumed
ba_data["ba_year_consumed"] = ba_data.groupby(["ba_code", "year"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["ba_code", "year"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# ba_year_generated
ba_data["ba_year_generated"] = ba_data.groupby(["ba_code", "year"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["ba_code", "year"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")

# region_hour_consumed
ba_data["region_hour_consumed"] = ba_data.groupby(
    ["region", "datetime_utc"], dropna=False
)["co2_emissions_consumed_lb"].transform("sum") / ba_data.groupby(
    ["region", "datetime_utc"], dropna=False
)["consumed_mwh"].transform("sum")
# region_hour_generated
ba_data["region_hour_generated"] = ba_data.groupby(
    ["region", "datetime_utc"], dropna=False
)["co2_emissions_generated_lb"].transform("sum") / ba_data.groupby(
    ["region", "datetime_utc"], dropna=False
)["positive_generation_mwh"].transform("sum")
# region_month_consumed
ba_data["region_month_consumed"] = ba_data.groupby(["region", "month"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["region", "month"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# region_month_generated
ba_data["region_month_generated"] = ba_data.groupby(["region", "month"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["region", "month"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# region_year_consumed
ba_data["region_year_consumed"] = ba_data.groupby(["region", "year"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["region", "year"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# region_year_generated
ba_data["region_year_generated"] = ba_data.groupby(["region", "year"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["region", "year"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")

# grid_hour_consumed
ba_data["grid_hour_consumed"] = ba_data.groupby(["grid", "datetime_utc"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["grid", "datetime_utc"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# grid_hour_generated
ba_data["grid_hour_generated"] = ba_data.groupby(
    ["grid", "datetime_utc"], dropna=False
)["co2_emissions_generated_lb"].transform("sum") / ba_data.groupby(
    ["grid", "datetime_utc"], dropna=False
)["positive_generation_mwh"].transform("sum")
# grid_month_consumed
ba_data["grid_month_consumed"] = ba_data.groupby(["grid", "month"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["grid", "month"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# grid_month_generated
ba_data["grid_month_generated"] = ba_data.groupby(["grid", "month"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["grid", "month"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# grid_year_consumed
ba_data["grid_year_consumed"] = ba_data.groupby(["grid", "year"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["grid", "year"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# grid_year_generated
ba_data["grid_year_generated"] = ba_data.groupby(["grid", "year"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["grid", "year"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")

# national_hour_consumed
ba_data["national_hour_consumed"] = ba_data.groupby(["datetime_utc"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["datetime_utc"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# national_hour_generated
ba_data["national_hour_generated"] = ba_data.groupby(["datetime_utc"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["datetime_utc"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# national_month_consumed
ba_data["national_month_consumed"] = ba_data.groupby(["month"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["month"], dropna=False)["consumed_mwh"].transform(
    "sum"
)
# national_month_generated
ba_data["national_month_generated"] = ba_data.groupby(["month"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["month"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# national_year_consumed
ba_data["national_year_consumed"] = ba_data.groupby(["year"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["year"], dropna=False)["consumed_mwh"].transform(
    "sum"
)
# national_year_generated
ba_data["national_year_generated"] = ba_data.groupby(["year"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["year"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")

In [ ]:
# calculate total emissions
emissions_calc = ba_data[["ba_code", "region", "datetime_utc", "demand_mwh"]].copy()
for spatial_level in ["ba", "region", "grid", "national"]:
    for temporal_level in ["hour", "month", "year"]:
        for type in ["consumed", "generated"]:
            emissions_calc[f"{spatial_level}_{temporal_level}_{type}"] = (
                ba_data[f"{spatial_level}_{temporal_level}_{type}"]
                * ba_data["demand_mwh"]
            )

# add US total to the dataframe
# us_total = (
#     emissions_calc.drop(columns=["ba_code"])
#     .groupby(["datetime_utc"])
#     .sum(numeric_only=True)
#     .reset_index()
# )
# us_total["ba_code"] = "US"
# emissions_calc = pd.concat([emissions_calc, us_total])


emissions_calc

In [ ]:
# calculate the annual error metrics for each ba and for the US total
annual_error_wide = (
    emissions_calc.drop(columns=["datetime_utc"])
    .groupby(["ba_code"])
    .sum(numeric_only=True)
)
annual_error = []
for ba in annual_error_wide.index:
    for spatial_level in ["ba", "grid", "national"]:
        for temporal_level in ["hour", "month", "year"]:
            for type in ["consumed", "generated"]:
                annual_error.append(
                    {
                        "ba_code": ba,
                        "spatial_level": spatial_level,
                        "temporal_level": temporal_level,
                        "type": type,
                        "demand": annual_error_wide.loc[ba, f"demand_mwh"],
                        "error": (
                            annual_error_wide.loc[
                                ba, f"{spatial_level}_{temporal_level}_{type}"
                            ]
                            - annual_error_wide.loc[ba, f"ba_hour_consumed"]
                        )
                        / annual_error_wide.loc[ba, f"ba_hour_consumed"],
                    }
                )

annual_error = pd.DataFrame(annual_error)
annual_error


In [ ]:
# Custom sort orders (fine → coarse / preferred → less preferred)
spatial_order = ["ba", "grid", "national"]
temporal_order = ["hour", "month", "year"]
type_order = ["consumed", "generated"]

annual_error["spatial_level"] = pd.Categorical(
    annual_error["spatial_level"], categories=spatial_order, ordered=True
)
annual_error["temporal_level"] = pd.Categorical(
    annual_error["temporal_level"], categories=temporal_order, ordered=True
)
annual_error["type"] = pd.Categorical(
    annual_error["type"], categories=type_order, ordered=True
)

# Reorder this list to test different ranking priorities, e.g.:
#   ["spatial_level", "temporal_level", "type"]
#   ["temporal_level", "spatial_level", "type"]
#   ["type", "spatial_level", "temporal_level"]
sort_by = ["temporal_level", "spatial_level", "type"]

annual_error.sort_values(["ba_code"] + sort_by)


In [ ]:
annual_error.groupby(["spatial_level", "temporal_level", "type"]).apply(
    lambda df: pd.Series(
        {
            "weighted_avg_abs_error": (
                (df["error"].abs() * df["demand"]).sum() / df["demand"].sum() * 100
            ).round(1)
        }
    )
).reset_index().sort_values(
    by="weighted_avg_abs_error", ascending=True
)  # ["type","spatial_level","temporal_level", ])

In [ ]:
annual_error.groupby(["spatial_level", "temporal_level", "type"]).apply(
    lambda df: pd.Series(
        {
            "weighted_avg_abs_error": (
                (df["error"].abs() * df["demand"]).sum() / df["demand"].sum() * 100
            ).round(1)
        }
    )
).reset_index().sort_values(
    by=[
        "spatial_level",
        "type",
        "temporal_level",
    ]
)

In [ ]:
# why are the hourly vs annual values almost the same?
annual_error[
    (annual_error["spatial_level"] == "ba")
    & (annual_error["type"] == "consumed")
    & (annual_error["ba_code"] == "CISO")
]

In [ ]:
ba_data[ba_data["ba_code"] == "NYIS"]

In [ ]:
emissions_calc.loc[
    emissions_calc["ba_code"] == "NYIS", ["ba_hour_consumed", "ba_year_consumed"]
].sum()

In [ ]:
data_to_plot = ba_data[ba_data["ba_code"] == "NYIS"].copy()
data_to_plot["ba_hour_consumed"] = data_to_plot["ba_hour_consumed"] * 35
px.line(data_to_plot, x="datetime_local", y=["demand_mwh", "ba_hour_consumed"])

In [ ]:
# calculate hourly MAPE
hourly_mape = emissions_calc.copy()
for spatial_level in ["ba", "grid", "national"]:
    for temporal_level in ["hour", "month", "year"]:
        for type in ["consumed", "generated"]:
            if f"{spatial_level}_{temporal_level}_{type}" == "ba_hour_consumed":
                continue
            else:
                hourly_mape[f"{spatial_level}_{temporal_level}_{type}"] = (
                    (
                        hourly_mape[f"{spatial_level}_{temporal_level}_{type}"]
                        - hourly_mape[f"ba_hour_consumed"]
                    )
                    / hourly_mape[f"ba_hour_consumed"]
                ).abs()
hourly_mape["ba_hour_consumed"] = 0.0
annual_mape_wide = (
    hourly_mape.drop(columns=["datetime_utc"])
    .groupby(["ba_code"])
    .mean(numeric_only=True)
)

annual_mape = []
for ba in annual_mape_wide.index:
    for spatial_level in ["ba", "grid", "national"]:
        for temporal_level in ["hour", "month", "year"]:
            for type in ["consumed", "generated"]:
                annual_mape.append(
                    {
                        "ba_code": ba,
                        "spatial_level": spatial_level,
                        "temporal_level": temporal_level,
                        "type": type,
                        "error": annual_mape_wide.loc[
                            ba, f"{spatial_level}_{temporal_level}_{type}"
                        ],
                    }
                )

annual_mape = pd.DataFrame(annual_mape)
annual_mape

In [ ]:
annual_mape[annual_mape["ba_code"] == "NYIS"].round(3)

# Local regions
Do "local" subdivisions provide meaningful gain in accuracy. 

Test 1: MISO
- Use data from MISO dashboard for LRZs, and LRZ demand data from EIA-930
- Use that as the source of truth, and compare to MISO-wide numbers

Test 2: EIA "regions" as proxy for large regions
- Examples: Northwest, Florida, Southwest, Carolinas
- multiply BA demand by regional EFs

## EIA Proxy

In [ ]:
# EIA proxies
region_proxy = emissions_calc[
    emissions_calc["region"].isin(["FLA", "CAR", "SW", "NW"])
].copy()

# calculate the annual error metrics for each ba and for the US total
annual_error_proxy_wide = (
    region_proxy.drop(columns=["datetime_utc"])
    .groupby(["ba_code", "region"])
    .sum()
    .reset_index()
)
annual_proxy_error = []
for ba in annual_error_proxy_wide["ba_code"].unique():
    for spatial_level in ["ba", "region"]:
        for temporal_level in ["hour", "month", "year"]:
            for type in ["consumed", "generated"]:
                annual_proxy_error.append(
                    {
                        "ba_code": ba,
                        "region": annual_error_proxy_wide.loc[
                            annual_error_proxy_wide["ba_code"] == ba, "region"
                        ].values[0],
                        "spatial_level": spatial_level,
                        "temporal_level": temporal_level,
                        "type": type,
                        "error": (
                            annual_error_proxy_wide.loc[
                                annual_error_proxy_wide["ba_code"] == ba,
                                f"{spatial_level}_{temporal_level}_{type}",
                            ].values[0]
                            - annual_error_proxy_wide.loc[
                                annual_error_proxy_wide["ba_code"] == ba,
                                f"ba_hour_consumed",
                            ].values[0]
                        )
                        / annual_error_proxy_wide.loc[
                            annual_error_proxy_wide["ba_code"] == ba,
                            f"ba_hour_consumed",
                        ].values[0],
                    }
                )

annual_proxy_error = pd.DataFrame(annual_proxy_error)
annual_proxy_error


In [ ]:
annual_proxy_error[
    (annual_proxy_error["temporal_level"] == "hour")
    & (annual_proxy_error["type"] == "consumed")
    & (annual_proxy_error["spatial_level"] == "region")
].sort_values(by="region", ascending=True)

In [ ]:
# calculate hourly MAPE
hourly_region_mape = region_proxy.copy()
for spatial_level in ["ba", "region"]:
    for temporal_level in ["hour", "month", "year"]:
        for type in ["consumed", "generated"]:
            if f"{spatial_level}_{temporal_level}_{type}" == "ba_hour_consumed":
                continue
            else:
                hourly_region_mape[f"{spatial_level}_{temporal_level}_{type}"] = (
                    (
                        hourly_region_mape[f"{spatial_level}_{temporal_level}_{type}"]
                        - hourly_region_mape[f"ba_hour_consumed"]
                    )
                    / hourly_region_mape[f"ba_hour_consumed"]
                ).abs()
hourly_region_mape["ba_hour_consumed"] = 0.0
annual_region_mape_wide = (
    hourly_region_mape.drop(columns=["datetime_utc"])
    .groupby(["ba_code"])
    .mean(numeric_only=True)
)

annual_region_mape = []
for ba in annual_region_mape_wide.index:
    for spatial_level in ["ba", "region"]:
        for temporal_level in ["hour", "month", "year"]:
            for type in ["consumed", "generated"]:
                annual_region_mape.append(
                    {
                        "ba_code": ba,
                        "spatial_level": spatial_level,
                        "temporal_level": temporal_level,
                        "type": type,
                        "error": annual_region_mape_wide.loc[
                            ba, f"{spatial_level}_{temporal_level}_{type}"
                        ],
                    }
                )

annual_region_mape = pd.DataFrame(annual_region_mape)
annual_region_mape

## MISO data
Get subregion demand data from 930

In [ ]:
# get subregion demand data from 930
subregion_cols = {
    "Subregion 0001": "1",
    "Subregion 0004": "4",
    "Subregion 0006": "6",
    "Subregion 0027": "2/7",
    "Subregion 0035": "3/5",
    "Subregion 8910": "8/9/10",
}
miso_subregion = pd.read_excel(
    f"https://www.eia.gov/electricity/gridmonitor/knownissues/xls/MISO.xlsx",
    sheet_name="Published Hourly Data",
    usecols=lambda col: col in (ba_columns_to_use + list(subregion_cols.keys())),
)
# filter data to a specific local year
miso_subregion = miso_subregion[miso_subregion["Local time"].dt.year == year]
# rename columns
miso_subregion.rename(columns=column_map, inplace=True)
miso_subregion.rename(columns=subregion_cols, inplace=True)
# pivot the data so subregion is a column, and demand is the value
miso_subregion = miso_subregion.melt(
    id_vars=["datetime_utc"],
    value_vars=list(subregion_cols.values()),
    var_name="subregion",
    value_name="demand",
)
miso_subregion["datetime_utc"] = pd.to_datetime(
    miso_subregion["datetime_utc"], utc=True
)
miso_subregion

In [ ]:
# Original documented API: consumedEmissionsEvents
# https://miso.singularity.energy/api-docs#/operations/consumedEmissionsEvents
#
# Example request (single LRZ):
#   GET https://api.miso.singularity.energy/v1/events/consumed/emissions
#     ?start=2025-01-01T00:00:00Z
#     &end=2025-01-02T00:00:00Z
#     &agg_type=lrz
#     &agg_ids=1
#     &units=us
#     &resolution=hourly
#   Headers: x-api-key: <API_KEY>
#
# Combined regions add combine=true, e.g. agg_ids=2,7&combine=true
# Footprint-level (agg_type=miso) omits agg_ids per the spec.

ENDPOINT = "https://api.miso.singularity.energy/v1/events/consumed/emissions"


# Query window in UTC. `start` is inclusive, `end` is exclusive.
start = "2025-01-01T00:00:00Z"
end = "2026-01-01T00:00:00Z"

agg_type = "lrz"
# keys: LRZ id(s) to query; values: whether to combine those ids into one event
agg_ids_combine = {
    "1": False,
    "2,7": True,
    "3,5": True,
    "4": False,
    "6": False,
    "8,9,10": True,
}
units = "us"  # "metric" (kg CO2e) or "us" (lbs CO2e)
chunk_days = 7  # API responses stay small; increase if queries are reliably fast

import os

API_KEY = None
api_key_filename = ".miso_api_key"
if os.path.exists(api_key_filename):
    with open(api_key_filename) as f:
        API_KEY = f.read().strip()


def _utc(ts):
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")


def fetch_consumed_emissions_api(
    start, end, agg_type="lrz", agg_ids=None, combine=False, units="us"
):
    """Query the documented consumedEmissionsEvents endpoint."""
    if not API_KEY:
        raise ValueError("Set API_KEY to your MISO Emissions API key.")

    start_ts, end_ts = _utc(start), _utc(end)
    events = []
    chunk_start = start_ts
    while chunk_start < end_ts:
        chunk_end = min(chunk_start + pd.Timedelta(days=chunk_days), end_ts)
        params = {
            "start": chunk_start.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "end": chunk_end.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "agg_type": agg_type,
            "units": units,
            "resolution": "hourly",
        }
        # Required unless agg_type is "miso".
        if agg_ids and agg_type != "miso":
            params["agg_ids"] = agg_ids
        if combine:
            params["combine"] = "true"
        response = requests.get(
            ENDPOINT,
            headers={
                "x-api-key": API_KEY,
                "User-Agent": "open-grid-emissions-notebook",
            },
            params=params,
            timeout=60,
        )
        if not response.ok:
            raise RuntimeError(
                f"{response.status_code} {response.reason} for "
                f"{params['start']}–{params['end']} agg_ids={agg_ids}: {response.text}"
            )
        events.extend(response.json().get("data", []))
        chunk_start = chunk_end

    if not events:
        return pd.DataFrame()

    df = pd.json_normalize(events)
    df.columns = [c.replace("emissions.", "") for c in df.columns]
    drop_cols = [
        c for c in df.columns if c.startswith("total_") or c.startswith("fuel_mix.")
    ]
    df = df.drop(columns=drop_cols, errors="ignore")
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["requested_agg_ids"] = agg_ids
    df["combine"] = combine
    return df


lrz_emissions_cache = f"lrz_emissions_{year}.csv.zip"

if os.path.exists(lrz_emissions_cache):
    print(f"Loading lrz_emissions from {lrz_emissions_cache}")
    lrz_emissions = pd.read_csv(lrz_emissions_cache)
    lrz_emissions["datetime_utc"] = pd.to_datetime(
        lrz_emissions["datetime_utc"], utc=True
    )
else:
    api_frames = []
    for agg_ids, combine in agg_ids_combine.items():
        api_frames.append(
            fetch_consumed_emissions_api(
                start=start,
                end=end,
                agg_type=agg_type,
                agg_ids=agg_ids,
                combine=combine,
                units=units,
            )
        )

    lrz_emissions = (
        pd.concat(api_frames, ignore_index=True)
        .sort_values(["timestamp", "requested_agg_ids", "agg_id"])
        .reset_index(drop=True)
    )
    lrz_emissions.rename(
        columns={
            "timestamp": "datetime_utc",
            "agg_id": "subregion",
            "rate_co2e_lbs_per_mwh": "local_hour_consumed",
        },
        inplace=True,
    )
    lrz_emissions["subregion"] = lrz_emissions["subregion"].map(
        {"1": "1", "4": "4", "6": "6", "2,7": "2/7", "3,5": "3/5", "10,8,9": "8/9/10"}
    )
    lrz_emissions = lrz_emissions[
        ["subregion", "datetime_utc", "local_hour_consumed"]
    ].sort_values(["subregion", "datetime_utc"])

    miso_emissions_api = fetch_consumed_emissions_api(
        start=start,
        end=end,
        agg_type="miso",
        combine=False,
        units=units,
    )
    miso_emissions_api.rename(
        columns={
            "timestamp": "datetime_utc",
            "agg_id": "subregion",
            "rate_co2e_lbs_per_mwh": "ba_hour_consumed",
        },
        inplace=True,
    )
    miso_emissions_api = miso_emissions_api[
        ["datetime_utc", "ba_hour_consumed"]
    ].sort_values(["datetime_utc"])

    lrz_emissions = lrz_emissions.drop_duplicates(
        subset=["subregion", "datetime_utc"], keep="first"
    )
    miso_emissions_api = miso_emissions_api.drop_duplicates(
        subset=["datetime_utc"], keep="first"
    )

    lrz_emissions = lrz_emissions.merge(
        miso_emissions_api, on="datetime_utc", how="left", validate="m:1"
    )
    lrz_emissions.to_csv(lrz_emissions_cache, index=False)
    print(f"Saved lrz_emissions to {lrz_emissions_cache}")

lrz_emissions

In [ ]:
lrz_emissions_calc = miso_subregion.merge(
    lrz_emissions, on=["subregion", "datetime_utc"], how="inner", validate="m:1"
)

lrz_emissions_calc["local_hour_consumed"] = (
    lrz_emissions_calc["local_hour_consumed"] * lrz_emissions_calc["demand"]
)
lrz_emissions_calc["ba_hour_consumed"] = (
    lrz_emissions_calc["ba_hour_consumed"] * lrz_emissions_calc["demand"]
)

# calculate the annual error metrics for each ba and for the US total
lrz_error_wide = (
    lrz_emissions_calc.drop(columns=["datetime_utc"]).groupby(["subregion"]).sum()
)
lrz_error = []
for ba in lrz_error_wide.index:
    for spatial_level in ["local", "ba"]:
        for temporal_level in ["hour"]:
            for type in ["consumed"]:
                lrz_error.append(
                    {
                        "ba_code": ba,
                        "spatial_level": spatial_level,
                        "temporal_level": temporal_level,
                        "type": type,
                        "error": (
                            lrz_error_wide.loc[
                                ba, f"{spatial_level}_{temporal_level}_{type}"
                            ]
                            - lrz_error_wide.loc[ba, f"local_hour_consumed"]
                        )
                        / lrz_error_wide.loc[ba, f"local_hour_consumed"],
                    }
                )

lrz_error = pd.DataFrame(lrz_error)
lrz_error[lrz_error["spatial_level"] == "ba"]

In [ ]:
lrz_emissions.groupby("subregion").mean(numeric_only=True)